### Load and preview the dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# Load dataset
data = load_breast_cancer()

# Convert to DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)

# Preview first five rows of data
print(df.head())

# blank line
print()

# View structure of the dataframe
print(df.info())

### Convert variables to "transactional" format

In [ ]:
# Convert numeric features into binary (above/below median)
df_binary = df.apply(lambda x: x > x.median())

# Convert True/False to 1/0
df_binary = df_binary.astype(int)

print(df_binary.head())

### Generate frequent itemsets

In [ ]:
from mlxtend.frequent_patterns import apriori

# Keep itemsets that occur at least 30% of the time
frequent_itemsets = apriori(df_binary, min_support=0.3, use_colnames=True)

print(frequent_itemsets.head())

### Generate the association rules

In [ ]:
from mlxtend.frequent_patterns import association_rules

# Keep rules where at least 70% of "A" transactions also contain "B" (the "reliable" rules)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

### Sort and output the association rules by lift value

In [ ]:
rules_sorted = rules.sort_values(by='lift', ascending=False)

print(rules_sorted.head())

### Visualize the association rules

#### Graphs are always optional, but help to understand the analysis

In [ ]:
# Scatter plot
plt.figure()
plt.scatter(rules['support'], rules['confidence'])

# Labels
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Support vs Confidence')

plt.show()

### Scatter plot with emphasis on lift value

In [ ]:
plt.figure()
plt.scatter(rules['support'], rules['confidence'], s=rules['lift']*50)

plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Support vs Confidence (Bubble Size = Lift)')

plt.show()

### Network Graph

In [ ]:
# Create network graph
G = nx.DiGraph()

# Add edges (rules)
for _, row in rules.head(10).iterrows():
    for antecedent in row['antecedents']:
        for consequent in row['consequents']:
            G.add_edge(antecedent, consequent, weight=row['lift'])

# Draw graph
plt.figure()
pos = nx.spring_layout(G)

nx.draw(G, pos, with_labels=True)
plt.title("Association Rules Network")

plt.show()

### Bar chart graph

In [ ]:
# Get top 5 rules by lift
top_rules = rules.sort_values(by='lift', ascending=False).head(5)

plt.figure()
plt.barh(range(len(top_rules)), top_rules['lift'])

plt.yticks(range(len(top_rules)), top_rules['antecedents'].astype(str))
plt.xlabel('Lift')
plt.title('Top Rules by Lift')

plt.show()